# Nemotron Submission Packaging + Tiny LS Update — 2026-04-28

Purpose:
- keep the known-good submission packaging flow
- add the smallest meaningful learning step on top of the adapter shell
- produce a valid `submission.zip` for Kaggle


In [8]:
import os
import sys
import site
import stat
import shutil
import zipfile
import random
import gc

import kagglehub
import torch
import torch.nn.functional as F
import polars as pl
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = "/kaggle/working"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "nemotron_lora_adapter")
ZIP_PATH = os.path.join(OUTPUT_DIR, "submission.zip")
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

TRAIN_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"

# -----------------------------
# 0) CUTLASS path
# -----------------------------
cutlass_candidates = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/",
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/",
]
for c in cutlass_candidates:
    if os.path.exists(c):
        site.addsitedir(c)
        print("Added CUTLASS path:", c)

# -----------------------------
# 1) Triton / Nemotron fixes from prior working SFT notebook
# -----------------------------
def _pure_rmsnorm_fn(
    x,
    weight,
    bias=None,
    z=None,
    eps=1e-5,
    group_size=None,
    norm_before_gate=True,
    upcast=True,
):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

utility_root_candidates = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script",
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script",
]
utility_root = None
for p in utility_root_candidates:
    if os.path.exists(p):
        utility_root = p
        break

print("Resolved utility root:", utility_root)

src = None
if utility_root is not None:
    candidate = os.path.join(
        utility_root, "triton", "backends", "nvidia", "bin", "ptxas-blackwell"
    )
    if os.path.exists(candidate):
        src = candidate

dst = "/tmp/ptxas-blackwell"
if src is not None:
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    print("Copied writable ptxas-blackwell to:", dst)
else:
    print("WARNING: ptxas-blackwell source not found")

# Standard ptxas if present
std_ptxas = None
if utility_root is not None:
    candidate = os.path.join(
        utility_root, "triton", "backends", "nvidia", "bin", "ptxas"
    )
    if os.path.exists(candidate):
        std_ptxas = candidate

if std_ptxas is not None:
    os.environ["TRITON_PTXAS_PATH"] = std_ptxas
    print("Using TRITON_PTXAS_PATH:", std_ptxas)

if os.path.exists(dst):
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    print("Using TRITON_PTXAS_BLACKWELL_PATH:", dst)

import triton.backends.nvidia.compiler as nv_compiler
nv_compiler.get_ptxas_version = lambda arch: "12.0"

if os.path.exists(ADAPTER_DIR):
    shutil.rmtree(ADAPTER_DIR)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("MODEL_PATH:", MODEL_PATH)
print("TRAIN_PATH:", TRAIN_PATH)
print("ADAPTER_DIR:", ADAPTER_DIR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch_dtype,
    trust_remote_code=True,
    device_map={"":0},
)

# Force slow path
patched_fast_path = False
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        mod.is_fast_path_available = False
        patched_fast_path = True
        print(f"Disabled fast path in module: {name}")

print("Fast path disabled:", patched_fast_path)

model.config.use_cache = False
try:
    model.gradient_checkpointing_disable()
    print("Gradient checkpointing disabled.")
except Exception as e:
    print("gradient_checkpointing_disable skipped:", e)

train = pl.read_csv(TRAIN_PATH)
train_subset = train.sample(n=min(4, train.height), seed=SEED, shuffle=True)
print("Tiny LS subset shape:", train_subset.shape)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Added CUTLASS path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/
Resolved utility root: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script
Copied writable ptxas-blackwell to: /tmp/ptxas-blackwell
Using TRITON_PTXAS_PATH: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas
Using TRITON_PTXAS_BLACKWELL_PATH: /tmp/ptxas-blackwell
MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
TRAIN_PATH: /kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv
ADAPTER_DIR: /kaggle/working/nemotron_lora_adapter


OutOfMemoryError: CUDA out of memory. Tried to allocate 31.60 GiB. GPU 0 has a total capacity of 94.97 GiB of which 6.52 GiB is free. Including non-PyTorch memory, this process has 88.45 GiB memory in use. Of the allocated memory 60.67 GiB is allocated by PyTorch, and 27.13 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
def build_text(row):
    prompt = row["prompt"]
    answer = str(row["answer"]).strip()

    user_text = (
        prompt
        + "\nPlease put your final answer inside \\boxed{}."
        + "\nFor example: \\boxed{your answer}"
        + "\nReturn only one final boxed answer."
    )
    assistant_text = f"\\boxed{{{answer}}}"

    return f"User: {user_text}\nAssistant: {assistant_text}"

texts = [build_text(r) for r in train_subset.to_dicts()]
print(texts[0][:400])

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4,
)

torch.cuda.empty_cache()
model.train()
losses = []

for step, text in enumerate(texts, start=1):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=False,
    )

    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}
    labels = enc["input_ids"].clone()

    optimizer.zero_grad()
    outputs = model(**enc, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

    val = float(loss.detach().cpu().item())
    losses.append(val)
    print(f"step={step}/{len(texts)} loss={val:.4f}")

print("Mean loss:", sum(losses) / len(losses))

model.save_pretrained(ADAPTER_DIR)
print("Saved files:", os.listdir(ADAPTER_DIR))

required = {"adapter_config.json", "adapter_model.safetensors"}
present = set(os.listdir(ADAPTER_DIR))
missing = required - present
if missing:
    raise ValueError(f"Missing required adapter files: {missing}")

del optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
required_files = ['adapter_config.json', 'adapter_model.safetensors']
for fname in required_files:
    full_path = os.path.join(ADAPTER_DIR, fname)
    if not os.path.exists(full_path):
        raise FileNotFoundError(f'Required file missing: {full_path}')

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        full_path = os.path.join(ADAPTER_DIR, fname)
        zf.write(full_path, arcname=fname)

print('Created:', ZIP_PATH)
print('Zip size (MB):', os.path.getsize(ZIP_PATH) / 1024 / 1024)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    print('Zip contents:', zf.namelist())
assert os.path.exists(ZIP_PATH), 'submission.zip was not created'
print('Done. This notebook is submission-only with tiny LS update.')
